# EDA: Subject-Level Aggregated Data

Exploratory data analysis on the subject-level aggregated dataset (N=28) for the
bachelor's thesis on ABPM hemodynamic coupling.

In [ ]:
import sys
import os

# Ensure CWD is project root so Config relative paths resolve correctly
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

from abpm_hemodynamic_coupling.config import Config, Columns
from abpm_hemodynamic_coupling.data_processing import DataLoader
from analysis.thesis_stats import (
    bootstrap_ci,
    format_median_iqr,
    format_mean_sd,
    derive_dipper_category,
    export_table,
)

config = Config()
THESIS_DIR = Path("results/thesis")
THESIS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300
sns.set_theme(style="whitegrid", font_scale=1.1)

In [ ]:
loader = DataLoader(config)
agg = loader.load_aggregated_data()

# Alias the typo column
if "avg_reaction_time_battey_2" in agg.columns:
    agg = agg.rename(columns={"avg_reaction_time_battey_2": "avg_reaction_time_battery_2"})

# Load pipeline results for joining
res = pd.read_csv(config.get_results_path(config.SUBJECT_METRICS_OUTPUT))
print(f"Aggregated: {len(agg)} subjects, {agg.shape[1]} columns")
print(f"Pipeline results: {len(res)} subjects")

## 1. Continuous Variable Distributions

In [ ]:
# Demographics distributions: age, bmi, height_cm, weight_kilos
demo_vars = ["age", "bmi", "height_cm", "weight_kilos"]
demo_labels = ["Age (years)", "BMI (kg/m^2)", "Height (cm)", "Weight (kg)"]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, (var, label) in enumerate(zip(demo_vars, demo_labels)):
    ax = axes[idx]
    data = agg[var].dropna()

    # Histogram with KDE
    sns.histplot(data, kde=True, ax=ax, color="steelblue", edgecolor="white", alpha=0.7)

    # Box plot as inset above the histogram
    ax_box = ax.inset_axes([0.0, 0.85, 1.0, 0.15])
    sns.boxplot(x=data, ax=ax_box, color="steelblue", fliersize=3)
    ax_box.set(xlabel="", ylabel="", xticks=[], yticks=[])
    ax_box.set_xlim(ax.get_xlim())
    for spine in ax_box.spines.values():
        spine.set_visible(False)

    # Shapiro-Wilk test
    w_stat, p_val = stats.shapiro(data)
    norm_str = "Normal" if p_val > 0.05 else "Non-normal"
    ax.annotate(
        f"W={w_stat:.3f}, p={p_val:.3f}\n({norm_str})",
        xy=(0.97, 0.70),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
    )
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(label)

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_eda_demographics.png", dpi=DPI, bbox_inches="tight")
plt.close()

In [ ]:
# Monitoring & sleep durations
dur_vars = ["monitoring_duration_m", "sleep_duration_h", "bp_load_%"]
dur_labels = ["Monitoring Duration (min)", "Sleep Duration (h)", "BP Load (%)"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (var, label) in enumerate(zip(dur_vars, dur_labels)):
    ax = axes[idx]
    data = agg[var].dropna()

    sns.histplot(data, kde=True, ax=ax, color="teal", edgecolor="white", alpha=0.7)

    ax_box = ax.inset_axes([0.0, 0.85, 1.0, 0.15])
    sns.boxplot(x=data, ax=ax_box, color="teal", fliersize=3)
    ax_box.set(xlabel="", ylabel="", xticks=[], yticks=[])
    ax_box.set_xlim(ax.get_xlim())
    for spine in ax_box.spines.values():
        spine.set_visible(False)

    w_stat, p_val = stats.shapiro(data)
    norm_str = "Normal" if p_val > 0.05 else "Non-normal"
    ax.annotate(
        f"W={w_stat:.3f}, p={p_val:.3f}\n({norm_str})",
        xy=(0.97, 0.70),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8),
    )
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(label)

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_eda_monitoring_sleep.png", dpi=DPI, bbox_inches="tight")
plt.close()

In [ ]:
# Dipping percentages with clinical thresholds
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

dip_vars = ["sbp_dip_%", "dbp_dip_%"]
dip_labels = ["SBP Dip (%)", "DBP Dip (%)"]
thresholds = [0, 10, 20]
category_labels = [
    ("Reverse\ndipper", -5),
    ("Non-\ndipper", 5),
    ("Normal\ndipper", 15),
    ("Extreme\ndipper", 25),
]

for idx, (var, label) in enumerate(zip(dip_vars, dip_labels)):
    ax = axes[idx]
    data = agg[var].dropna()

    sns.histplot(data, kde=True, ax=ax, color="coral", edgecolor="white", alpha=0.7, bins=10)

    # Clinical threshold lines
    for thr in thresholds:
        ax.axvline(thr, color="red", linestyle="--", linewidth=1, alpha=0.7)
        ax.text(
            thr,
            ax.get_ylim()[1] * 0.95,
            f"{thr}%",
            ha="center",
            va="top",
            fontsize=8,
            color="red",
        )

    # Category region annotations
    ylim = ax.get_ylim()
    for cat_label, x_pos in category_labels:
        if ax.get_xlim()[0] <= x_pos <= ax.get_xlim()[1]:
            ax.text(
                x_pos,
                ylim[1] * 0.5,
                cat_label,
                ha="center",
                va="center",
                fontsize=7,
                alpha=0.5,
                style="italic",
            )

    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(label)

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_eda_dipping.png", dpi=DPI, bbox_inches="tight")
plt.close()

In [ ]:
# Cognitive task variables: Stroop effects and reaction times
cog_stroop = pd.DataFrame(
    {
        "Battery 1": agg["stroop_effect_battery_1"].dropna(),
        "Battery 2": agg["stroop_effect_battery_2"].dropna(),
    }
)
cog_rt = pd.DataFrame(
    {
        "Battery 1": agg["avg_reaction_time_battery_1"].dropna(),
        "Battery 2": agg["avg_reaction_time_battery_2"].dropna(),
    }
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Stroop effects
stroop_melted = cog_stroop.melt(var_name="Battery", value_name="Stroop Effect (ms)")
sns.violinplot(
    data=stroop_melted,
    x="Battery",
    y="Stroop Effect (ms)",
    ax=axes[0],
    palette=["steelblue", "coral"],
    inner="box",
    cut=0,
)
sns.stripplot(
    data=stroop_melted,
    x="Battery",
    y="Stroop Effect (ms)",
    ax=axes[0],
    color="black",
    size=3,
    alpha=0.5,
)
axes[0].set_title("Stroop Effect by Battery")
axes[0].axhline(0, color="grey", linestyle="--", linewidth=0.8)

# Reaction times
rt_melted = cog_rt.melt(var_name="Battery", value_name="Avg Reaction Time (ms)")
sns.violinplot(
    data=rt_melted,
    x="Battery",
    y="Avg Reaction Time (ms)",
    ax=axes[1],
    palette=["steelblue", "coral"],
    inner="box",
    cut=0,
)
sns.stripplot(
    data=rt_melted,
    x="Battery",
    y="Avg Reaction Time (ms)",
    ax=axes[1],
    color="black",
    size=3,
    alpha=0.5,
)
axes[1].set_title("Average Reaction Time by Battery")

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_eda_cognitive.png", dpi=DPI, bbox_inches="tight")
plt.close()

In [ ]:
# Task stress ratios: grouped box plots
ratio_cols = [
    "SBP_cog_1_ratio", "DBP_cog_1_ratio", "HR_cog_1_ratio",
    "SBP_cog_2_ratio", "DBP_cog_2_ratio", "HR_cog_2_ratio",
    "SBP_phys_1_ratio", "DBP_phys_1_ratio", "HR_phys_1_ratio",
    "SBP_phys_2_ratio", "DBP_phys_2_ratio", "HR_phys_2_ratio",
]

# Parse ratio columns into structured long format
ratio_records = []
for col in ratio_cols:
    parts = col.replace("_ratio", "").split("_")
    measure = parts[0]  # SBP, DBP, HR
    task_type = "Cognitive" if parts[1] == "cog" else "Physical"
    battery = f"Battery {parts[2]}"
    for val in agg[col].dropna():
        ratio_records.append(
            {
                "Measure": measure,
                "Task Type": task_type,
                "Battery": battery,
                "Ratio": val,
            }
        )

ratio_df = pd.DataFrame(ratio_records)

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for idx, battery in enumerate(["Battery 1", "Battery 2"]):
    ax = axes[idx]
    subset = ratio_df[ratio_df["Battery"] == battery]
    sns.boxplot(
        data=subset,
        x="Measure",
        y="Ratio",
        hue="Task Type",
        ax=ax,
        palette={"Cognitive": "steelblue", "Physical": "coral"},
        fliersize=3,
    )
    ax.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.6, label="No change")
    ax.set_title(battery)
    ax.set_ylabel("Task / Pre-task Ratio" if idx == 0 else "")
    ax.set_xlabel("Hemodynamic Measure")
    if idx == 1:
        ax.legend(title="Task Type")
    else:
        ax.get_legend().remove()

fig.suptitle("Task Performance Ratios (Task / Pre-task)", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(
    THESIS_DIR / "fig_08_task_performance_distributions.png", dpi=DPI, bbox_inches="tight"
)
plt.close()

## 2. Categorical Variable Summaries

In [ ]:
# Cohort composition: horizontal bar chart of binary variable proportions
binary_vars = [
    "is_female",
    "is_young",
    "monitoring_diagnosis_is_healthy",
    "has_chronic_diseases",
    "bmi_is_normal",
    "sleep_less_than_7h",
    "is_satisfied_with_sleep",
    "air_alert_during_monitoring",
    "is_normal_dipper_by_sbp",
    "is_normal_dipper_by_dbp",
    "working_under_stress",
    "has_mental_labor",
    "not_smoking",
    "drinks_coffee",
    "food_quality_is_good",
    "partnership_is_registered",
    "has_university_degree",
    "sleep_before_midnight",
]

proportions = agg[binary_vars].mean().sort_values()
counts = agg[binary_vars].sum().reindex(proportions.index)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(range(len(proportions)), proportions.values, color="steelblue", edgecolor="white")
ax.set_yticks(range(len(proportions)))
ax.set_yticklabels(
    [v.replace("_", " ").title() for v in proportions.index], fontsize=9
)
ax.set_xlabel("Proportion")
ax.set_title("Cohort Composition: Binary Variables")
ax.set_xlim(0, 1.15)

# Annotate with count / total
for i, (prop, count) in enumerate(zip(proportions.values, counts.values)):
    ax.text(
        prop + 0.02,
        i,
        f"{int(count)}/{len(agg)} ({prop:.0%})",
        va="center",
        fontsize=8,
    )

ax.axvline(0.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.5)
plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_eda_binary_vars.png", dpi=DPI, bbox_inches="tight")
plt.close()

In [ ]:
# Key cross-tabulations as annotated heatmaps
cross_tabs = [
    ("gender", "is_young", "Gender x Age Group"),
    ("monitoring_diagnosis_is_healthy", "has_chronic_diseases", "Healthy Dx x Chronic Diseases"),
    ("sleep_less_than_7h", "is_satisfied_with_sleep", "Short Sleep x Satisfaction"),
    ("is_normal_dipper_by_sbp", "is_normal_dipper_by_dbp", "SBP Dipper x DBP Dipper"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, (row_var, col_var, title) in enumerate(cross_tabs):
    ax = axes[idx]
    ct = pd.crosstab(agg[row_var], agg[col_var])
    sns.heatmap(
        ct,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        cbar=False,
        linewidths=0.5,
    )
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(col_var.replace("_", " ").title())
    ax.set_ylabel(row_var.replace("_", " ").title())

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_eda_crosstabs.png", dpi=DPI, bbox_inches="tight")
plt.close()

In [ ]:
# Dipping categories: derive 4-level categories, cross-tabulate SBP vs DBP
agg["sbp_dipper_cat"] = derive_dipper_category(agg["sbp_dip_%"])
agg["dbp_dipper_cat"] = derive_dipper_category(agg["dbp_dip_%"])

print("SBP dipper categories:")
print(agg["sbp_dipper_cat"].value_counts().sort_index())
print("\nDBP dipper categories:")
print(agg["dbp_dipper_cat"].value_counts().sort_index())

ct_dip = pd.crosstab(agg["sbp_dipper_cat"], agg["dbp_dipper_cat"])
print("\nCross-tabulation (SBP rows x DBP cols):")
print(ct_dip)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    ct_dip,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Dipper Category Agreement: SBP vs DBP")
ax.set_xlabel("DBP Dipper Category")
ax.set_ylabel("SBP Dipper Category")

# Agreement count
agree = sum(ct_dip.loc[cat, cat] for cat in ct_dip.index if cat in ct_dip.columns)
total = ct_dip.values.sum()
ax.text(
    0.5,
    -0.12,
    f"Agreement: {agree}/{total} ({agree / total:.0%})",
    transform=ax.transAxes,
    ha="center",
    fontsize=11,
)

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_eda_dipper_agreement.png", dpi=DPI, bbox_inches="tight")
plt.close()

## 3. Missing Data Analysis

In [ ]:
# Missingness heatmap: focus on columns with any missing values
missing_pct = agg.isnull().mean().sort_values(ascending=False)
cols_with_missing = missing_pct[missing_pct > 0].index.tolist()

if cols_with_missing:
    miss_matrix = agg[cols_with_missing].isnull().astype(int)

    fig, ax = plt.subplots(figsize=(max(8, len(cols_with_missing) * 0.6), 8))
    sns.heatmap(
        miss_matrix,
        cmap=["#f0f0f0", "#d32f2f"],
        cbar_kws={"label": "Missing", "ticks": [0, 1]},
        ax=ax,
        yticklabels=[f"S{i+1}" for i in range(len(agg))],
        xticklabels=[c.replace("_", "\n") for c in cols_with_missing],
    )
    ax.set_title("Missing Data Pattern (red = missing)")
    ax.set_ylabel("Subject")

    plt.tight_layout()
    fig.savefig(THESIS_DIR / "fig_eda_missing_heatmap.png", dpi=DPI, bbox_inches="tight")
    plt.close()
else:
    print("No missing data found.")

In [ ]:
# Missing data summary table
missing_summary = pd.DataFrame(
    {
        "Column": missing_pct.index,
        "Missing Count": agg.isnull().sum().reindex(missing_pct.index).values,
        "Missing %": (missing_pct.values * 100).round(1),
    }
)
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].reset_index(drop=True)

if len(missing_summary) > 0:
    print(f"Columns with missing values: {len(missing_summary)}")
    print(missing_summary.to_string(index=False))

    # Check pattern: do subjects missing ratio cols also miss stage durations?
    ratio_miss_cols = [c for c in cols_with_missing if "ratio" in c]
    stage_miss_cols = [c for c in cols_with_missing if "duration" in c or "stage" in c]
    if ratio_miss_cols:
        any_ratio_missing = agg[ratio_miss_cols].isnull().any(axis=1)
        print(f"\nSubjects missing any task ratio: {any_ratio_missing.sum()}")
    if stage_miss_cols:
        any_stage_missing = agg[stage_miss_cols].isnull().any(axis=1)
        print(f"Subjects missing any stage duration: {any_stage_missing.sum()}")
else:
    print("No missing data across all columns.")

## 4. Normality Assessment

In [ ]:
# Shapiro-Wilk tests on key continuous variables + Q-Q plots
normality_vars = [
    "age", "bmi", "height_cm", "weight_kilos",
    "monitoring_duration_m", "sleep_duration_h", "bp_load_%",
    "sbp_dip_%", "dbp_dip_%",
    "stroop_effect_battery_1", "stroop_effect_battery_2",
    "avg_reaction_time_battery_1", "avg_reaction_time_battery_2",
    "SBP_all_med", "DBP_all_med", "HR_all_med",
    "SBP_all_iqr", "DBP_all_iqr", "HR_all_iqr",
]

# Summary table
normality_results = []
for var in normality_vars:
    data = agg[var].dropna()
    if len(data) >= 3:
        w_stat, p_val = stats.shapiro(data)
        normality_results.append(
            {
                "Variable": var,
                "N": len(data),
                "W": round(w_stat, 4),
                "p-value": round(p_val, 4),
                "Normal (p>0.05)": "Yes" if p_val > 0.05 else "No",
            }
        )

norm_df = pd.DataFrame(normality_results)
print(norm_df.to_string(index=False))
export_table(norm_df, THESIS_DIR / "table_normality")

# Q-Q plots grid
n_vars = len(normality_vars)
n_cols = 4
n_rows = (n_vars + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.ravel()

for idx, var in enumerate(normality_vars):
    ax = axes[idx]
    data = agg[var].dropna().values
    if len(data) >= 3:
        stats.probplot(data, dist="norm", plot=ax)
        ax.set_title(var.replace("_", " "), fontsize=9)
        ax.get_lines()[0].set_markersize(4)
    else:
        ax.set_visible(False)

# Hide unused axes
for idx in range(n_vars, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Q-Q Plots: Normality Assessment", fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_S1_qq_plots.png", dpi=DPI, bbox_inches="tight")
plt.close()

## 5. Subject-Level Correlation Heatmap

In [ ]:
# Hypothesis-driven correlations: predictors vs pipeline outcomes
predictor_cols = [
    "age", "bmi", "bp_load_%", "sbp_dip_%", "dbp_dip_%",
    "sleep_duration_h", "DBP_all_med", "DBP_all_iqr",
]
outcome_cols = [
    "DBP_Cognitive Task_DeltaBias",
    "DBP_Cognitive Task_Anomaly",
    "DBP_Physical Task_DeltaBias",
    "DBP_Physical Task_Anomaly",
    "DBP_Air Alert_DeltaBias",
    "DBP_Air Alert_Anomaly",
]

# Merge agg and res on participant_id
merged = agg.merge(res, on="participant_id", how="inner")
print(f"Merged dataset: {len(merged)} subjects")

# Compute Spearman correlations
all_cols = predictor_cols + outcome_cols
corr_data = merged[all_cols].dropna()
n_corr = len(corr_data)

corr_matrix = corr_data[predictor_cols + outcome_cols].corr(method="spearman")
# Extract the cross-correlation block: predictors (rows) x outcomes (cols)
cross_corr = corr_matrix.loc[predictor_cols, outcome_cols]

# Compute p-values for each pair
p_matrix = pd.DataFrame(
    np.ones((len(predictor_cols), len(outcome_cols))),
    index=predictor_cols,
    columns=outcome_cols,
)
for pred in predictor_cols:
    for out in outcome_cols:
        pair_data = merged[[pred, out]].dropna()
        if len(pair_data) >= 3:
            rho, p_val = stats.spearmanr(pair_data[pred], pair_data[out])
            p_matrix.loc[pred, out] = p_val

# Build significance annotation matrix
annot_matrix = cross_corr.round(2).astype(str)
for pred in predictor_cols:
    for out in outcome_cols:
        p_val = p_matrix.loc[pred, out]
        rho_str = f"{cross_corr.loc[pred, out]:.2f}"
        if p_val < 0.001:
            annot_matrix.loc[pred, out] = rho_str + "***"
        elif p_val < 0.01:
            annot_matrix.loc[pred, out] = rho_str + "**"
        elif p_val < 0.05:
            annot_matrix.loc[pred, out] = rho_str + "*"
        else:
            annot_matrix.loc[pred, out] = rho_str

# Clean labels for display
row_labels = [c.replace("_", " ") for c in predictor_cols]
col_labels = [
    c.replace("DBP_", "").replace("_", " ") for c in outcome_cols
]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    cross_corr.values.astype(float),
    annot=annot_matrix.values,
    fmt="",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    xticklabels=col_labels,
    yticklabels=row_labels,
    linewidths=0.5,
    cbar_kws={"label": "Spearman rho"},
)
ax.set_title(f"Subject-Level Spearman Correlations (n={n_corr})\n* p<.05  ** p<.01  *** p<.001")
plt.tight_layout()
fig.savefig(
    THESIS_DIR / "fig_05_subject_correlation_heatmaps.png", dpi=DPI, bbox_inches="tight"
)
plt.close()

## 6. Export Clean Dataset

In [ ]:
# Ensure derived categories are present (may already be set above)
agg["sbp_dipper_cat"] = derive_dipper_category(agg["sbp_dip_%"])
agg["dbp_dipper_cat"] = derive_dipper_category(agg["dbp_dip_%"])
agg.to_csv(THESIS_DIR / "aggregated_clean.csv", index=False)
print(f"Exported: {agg.shape}")